In [1]:
pip install PyQt6 SpeechRecognition PyAudio


   ---------------------------------------- 0.0/6.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/6.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/6.8 MB ? eta -:--:--
   --- ------------------------------------ 0.5/6.8 MB 514.7 kB/s eta 0:00:13
   --- ------------------------------------ 0.5/6.8 MB 514.7 kB/s eta 0:00:13
   ---- ----------------------------------- 0.8/6.8 MB 647.6 kB/s eta 0:00:10
   ------ --------------------------------- 1.0/6.8 MB 771.5 kB/s eta 0:00:08
   ------ --------------------------------- 1.0/6.8 MB 771.5 kB/s eta 0:00:08
   ------ --------------------------------- 1.0/6.8 MB 771.5 kB/s eta 0:00:08
   ------ --------------------------------- 1.0/6.8 MB 771.5 kB/s eta 0:00:08
   ------- -------------------------------- 1.3/6.8 MB 579.2 kB/s eta 0:00:10
   ------- -------------------------------- 1.3/6.8 MB 579.2 kB/s eta 0:00:10
   --------- ---------


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Exception:
Traceback (most recent call last):
  File "c:\Users\Nicholas\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "c:\Users\Nicholas\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ~~~~~~~~~~~~~^^^^^
  File "c:\Users\Nicholas\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ~~~~~~~~~~~~~^^^^^
  File "c:\Users\Nicholas\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 98, in read
    data: bytes = self.__fp.read(amt)
                  ~~~

In [2]:
pip install git+https://github.com/m-bain/whisperx.git --default-timeout=1000

  Cloning https://github.com/m-bain/whisperx.git to c:\users\nicholas\appdata\local\temp\pip-req-build-jregdzwg
Note: you may need to restart the kernel to use updated packages.


  Running command git clone --filter=blob:none --quiet https://github.com/m-bain/whisperx.git 'C:\Users\Nicholas\AppData\Local\Temp\pip-req-build-jregdzwg'
  error: RPC failed; curl 56 schannel: server closed abruptly (missing close_notify)
  error: 8879 bytes of body are still expected
  fetch-pack: unexpected disconnect while reading sideband packet
  fatal: early EOF
  fatal: index-pack failed
  fatal: could not fetch d517cfb81698d3f82fd1ceac29c22e994e191764 from promisor remote
  You can inspect what was checked out with 'git status'
  and retry with 'git restore --source=HEAD :/'

  error: subprocess-exited-with-error
  
  × git clone --filter=blob:none --quiet https://github.com/m-bain/whisperx.git 'C:\Users\Nicholas\AppData\Local\Temp\pip-req-build-jregdzwg' did not run successfully.
  │ exit code: 128
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1

In [3]:
import whisperx

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

device = "cuda" 
audio_file = "audio.mp3"
batch_size = 16 # adjust based on GPU VRAM
compute_type = "float16" # or "int8" for CPU

# 1. Load Model (The "Input" Pipeline)
model = whisperx.load_model("large-v2", device, compute_type=compute_type)

# 2. Process Audio
audio = whisperx.load_audio(audio_file)
result = model.transcribe(audio, batch_size=batch_size)

# 3. Alignment (For word-level timestamps, which Scriberr uses)
model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)

# 4. Diarization (Speaker identification)
diarize_model = whisperx.DiarizationPipeline(use_auth_token="YOUR_HF_TOKEN", device=device)
diarize_segments = diarize_model(audio)
result = whisperx.assign_word_speakers(diarize_segments, result)

print(result["segments"]) # The "Output"

ModuleNotFoundError: No module named 'whisperx'

In [ ]:
import sys
import speech_recognition as sr
from PyQt6.QtWidgets import QApplication, QWidget, QVBoxLayout, QPushButton, QTextEdit
from PyQt6.QtCore import QThread, pyqtSignal

class SpeechThread(QThread):
    text_received = pyqtSignal(str)

    def __init__(self):
        super().__init__()
        self.is_running = True
        self.recognizer = sr.Recognizer()
        self.microphone = sr.Microphone()

    def run(self):
        with self.microphone as source:
            # Adjust for ambient noise once at the start
            self.recognizer.adjust_for_ambient_noise(source, duration=1)
            
            while self.is_running:
                try:
                    # phrase_time_limit keeps the "realtime" feel snappy
                    audio = self.recognizer.listen(source, phrase_time_limit=3)
                    text = self.recognizer.recognize_google(audio)
                    self.text_received.emit(text)
                except sr.UnknownValueError:
                    continue  # Could not understand audio
                except sr.RequestError:
                    self.text_received.emit("[Error: Check Connection]")
                except Exception as e:
                    if self.is_running:
                        print(f"Error: {e}")

    def stop(self):
        self.is_running = False
        self.quit()

class AudioWindow(QWidget):
    def __init__(self):
        super().__init__()
        self.init_ui()
        self.speech_thread = None

    def init_ui(self):
        self.setWindowTitle("Live Scribe")
        self.setFixedSize(300, 400)
        
        layout = QVBoxLayout()

        self.output_area = QTextEdit()
        self.output_area.setReadOnly(True)
        self.output_area.setPlaceholderText("Transcribed text will appear here...")
        layout.addWidget(self.output_area)

        self.record_btn = QPushButton("Start Recording")
        self.record_btn.setStyleSheet("background-color: #2ecc71; color: white; font-weight: bold; padding: 10px;")
        self.record_btn.clicked.connect(self.toggle_recording)
        layout.addWidget(self.record_btn)

        self.setLayout(layout)

    def toggle_recording(self):
        if self.speech_thread and self.speech_thread.isRunning():
            # Stopping
            self.speech_thread.stop()
            self.record_btn.setText("Start Recording")
            self.record_btn.setStyleSheet("background-color: #2ecc71; color: white; font-weight: bold; padding: 10px;")
        else:
            # Starting
            self.speech_thread = SpeechThread()
            self.speech_thread.text_received.connect(self.update_text)
            self.speech_thread.start()
            self.record_btn.setText("Stop Recording")
            self.record_btn.setStyleSheet("background-color: #e74c3c; color: white; font-weight: bold; padding: 10px;")

    def update_text(self, text):
        self.output_area.append(f"> {text}")

if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = AudioWindow()
    window.show()
    sys.exit(app.exec())